(rd.match)=
# 3. Matching the Dispersion Suppressor to the Straight Section

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

## Introduction

The periodic beta functions in the arcs are different from the periodic beta functions in the
straight sections. After suppressing the dispersion to zero, four quadrupoles after the
dispersion suppressor are needed to match $\alpha_1$, $\alpha_2$, $\beta_1$, and $\beta_2$ to the
periodic ones of the following straight section.

```{figure} figures/match.svg
:width: 100%

Dispersion suppressor and matching section.
```

## Example: Forward Matching Section

The procedure for creating the forward matching section is shown below. Matching the reverse
matching section is left as an exercise.

1. Start with the lattice of [chapter 2](dispersion-suppressor.md), which contains the
   dispersion suppressors, and the straight section cells of chapter 1 (exercises 2 and 3):

In [2]:
from lattices.ch02_dispersion_suppressor import *

2. Define four quadrupoles for matching to the straight section. Because this is the forward
   matching section (F), and these are the 2nd and 3rd special focusing and defocusing quadrupoles,
   they are named `qff2`, `qdf2`, `qff3`, and `qdf3`. They start with the strength of the straight
   section quadrupoles:

In [3]:
k_ss = qfss.Kn1  # The straight section strength, to start with

with sb.elements():
    qff2 = sb.Quadrupole(L=0.5, Kn1=k_ss)
    qdf2 = sb.Quadrupole(L=0.5, Kn1=-k_ss)
    qff3 = sb.Quadrupole(L=0.5, Kn1=k_ss)
    qdf3 = sb.Quadrupole(L=0.5, Kn1=-k_ss)

3. Define the matching section `mssf` of the figure, with the straight section drifts `db`, and the
   line `arc_to_ssf` from the end of the arc to the straight section, which contains the
   dispersion suppressor and the matching section. In
   {jltext}`Julia, `{jl}`[a; b]`{jltext}` joins the vectors `{pytext}`Python, `{py}`a + b`{pytext}` joins the lists `{jl}`a`{py}`a`
   and {jl}`b`{py}`b`:

In [4]:
mssf = [qff2, d1, db, d2, qdf2, d1, db, d2, qff3, d1, db, d2, qdf3, d1, db, d2]
arc_to_ssf = dispsupf + mssf
line_f = sb.Beamline(arc_to_ssf, species_ref=electron, pc_ref=pc_ref)
tw_f = sb.TwissExpr(line_f, a_initial=tw_af.a[-1])
tw_f

TwissExpr of a Beamline with 32 elements (a_initial=a[end])

4. The last four quadrupoles need to be matched so that at the end of `arc_to_ssf` the Twiss
   parameters are equal to the periodic ones of the straight section FODO cells. These are
   computed by the `TwissExpr` `tw_ssf` of the straight section cell (chapter 1, exercise 2): its
   Twiss parameters at the beginning of the cell are
   {jl}`tw_ssf.beta1[begin]`{py}`tw_ssf.beta1[sb.begin]` and so on. The targets
   therefore compare two quantities, one of each `TwissExpr`:

In [5]:
targets = [tw_f.beta1[-1] == tw_ssf.beta1[sb.begin], tw_f.alpha1[-1] == tw_ssf.alpha1[sb.begin],
           tw_f.beta2[-1] == tw_ssf.beta2[sb.begin], tw_f.alpha2[-1] == tw_ssf.alpha2[sb.begin]]
targets

[Target: beta1[end] - beta1[qfss#1] == 0.0,
 Target: alpha1[end] - alpha1[qfss#1] == 0.0,
 Target: beta2[end] - beta2[qfss#1] == 0.0,
 Target: alpha2[end] - alpha2[qfss#1] == 0.0]

   No number has to be copied from one lattice to another: if the straight section cells change,
   these targets follow.

5. Run the match. A matching problem like this one can have several solutions: the knobs that make
   the targets hold are not unique. Which solution is found depends on the starting point and on
   the method of the optimizer. From the strengths of the straight section,
   {jl}`match!`{py}`sb.match` finds the
   solution of the Bmad version of this tutorial:

In [6]:
sb.match(vary=[sb.Knob(q, "Kn1") for q in (qff2, qdf2, qff3, qdf3)],
         targets=targets)

Match converged (all targets met): 5 iterations, 6 evaluations, 2.32 s
 Target                        value             goal    met
 beta1[end] - beta1[qfss#1]    -3.774758284e-11  == 0.0  yes
 alpha1[end] - alpha1[qfss#1]  4.991562719e-13   == 0.0  yes
 beta2[end] - beta2[qfss#1]    -6.065370428e-12  == 0.0  yes
 alpha2[end] - alpha2[qfss#1]  3.099631662e-12   == 0.0  yes
 Knob      before         after          lower  upper
 qff2.Kn1  0.3519574526   0.3568781026   -Inf   Inf
 qdf2.Kn1  -0.3519574526  -0.3527483412  -Inf   Inf
 qff3.Kn1  0.3519574526   0.3788662921   -Inf   Inf
 qdf3.Kn1  -0.3519574526  -0.3612538547  -Inf   Inf


   The Levenberg-Marquardt method finds another solution (exercise 3).

6. The betas through the line, from the arc into the straight section:

In [7]:
from matplotlib.figure import Figure
s = sb.value(tw_f.s[:])
fig = Figure(figsize=(7, 3.5), layout="constrained")
ax, ax2 = fig.subplots(2, 1)
ax.plot(s, sb.value(tw_f.beta1[:]), label="β₁")
ax.plot(s, sb.value(tw_f.beta2[:]), label="β₂")
ax.set_ylabel("β [m]")
ax.legend(loc="upper left")
ax2.plot(s, sb.value(tw_f.dx[:]))
ax2.set_xlabel("s [m]")
ax2.set_ylabel("η [m]")
fig

<Figure size 700x350 with 2 Axes>

   and at the end, together with the periodic values of the straight section:

In [8]:
sb.value([tw_f.beta1[-1], tw_ssf.beta1[sb.begin], tw_f.alpha2[-1], tw_ssf.alpha2[sb.begin]])

array([27.2059882 , 27.2059882 ,  0.48460556,  0.48460556])

## Exercises

1. **Reverse matching section:** Construct the reverse matching section `mssr` with the quadrupoles
   `qfr2`, `qdr2`, `qfr3`, `qdr3`, following the same steps as in the example but for the reverse
   cells (it follows the reverse dispersion suppressor, and matches into the reverse straight
   section cell). How do your quadrupole strengths compare to those of the forward matching
   section?

2. **Weights:** Optimization success or failure can depend heavily on the relative weights of the
   targets. As an example of this, start again with the unoptimized quadrupoles, and this time give
   the first target the weight `1e5`, by writing it as
   {jl}`Target(tw_f.beta1[end] - tw_ssf.beta1[begin], 0; weight=1e5)`{py}`sb.Target(tw_f.beta1[-1] - tw_ssf.beta1[sb.begin], 0, weight=1e5)`.
   With this, how well does the
   optimization do?

3. **Several solutions:** Start again with the unoptimized quadrupoles, and run the match with
   the Levenberg-Marquardt method
   ({jl}`solver=:levenberg_marquardt`{py}`solver="levenberg_marquardt"`). Check that the solution is
   different, and that it meets the targets. Compare the maximum beta functions in the matching
   section for both solutions, for example with
   {jl}`value(maximum(tw_f.beta1["qff2:qdf3"]))`{py}`sb.value(sb.maximum(tw_f.beta1["qff2:qdf3"]))`.
   Which
   would you choose?

The solution of exercise 1 is in the lattice file of this chapter,
[`lattices/ch03_match_to_straight.py`](https://github.com/bmad-sim/SciBmad.jl/tree/main/docs/src/ring-design/lattices/ch03_match_to_straight.py).